# LangGraph - Complete Guide

## A Comprehensive Tutorial on Building Complex AI Applications with LangGraph

---

## Table of Contents
1. Introduction to LangGraph
2. Core Concepts
3. Installation & Setup
4. Building Your First Graph
5. State Management
6. Control Flow Patterns
7. Agent Architecture
8. Advanced Features
9. Best Practices
10. Real-World Examples

# 1. Introduction to LangGraph

## What is LangGraph?

LangGraph is a library for building stateful, multi-actor applications with large language models (LLMs). It provides a framework to orchestrate complex workflows where:

- **Multiple agents** can interact and collaborate
- **State is managed** across execution steps
- **Cycles and loops** are naturally supported
- **Execution flow** can branch, merge, and conditionally route

## Key Advantages

1. **Cycle Support**: Unlike traditional DAGs (Directed Acyclic Graphs), LangGraph supports cycles, enabling agents to reason and iterate
2. **State Management**: Built-in mechanisms for managing and transforming state across the graph
3. **Streaming**: First-class support for streaming updates and intermediate results
4. **Debuggability**: Easy to inspect and debug complex workflows
5. **Flexibility**: Write custom logic in Python while leveraging LLMs

## Use Cases

- Multi-agent systems and conversations
- Iterative problem-solving workflows
- Research and planning tasks
- Approval workflows with human-in-the-loop
- Complex decision-making pipelines

# 2. Core Concepts

## Graphs

A **graph** in LangGraph is a network of nodes connected by edges:

- **Nodes**: Represent computation units (functions, LLM calls, etc.)
- **Edges**: Represent transitions between nodes
- **State**: Shared data passed between nodes

## State

State is the central concept in LangGraph:
- Defines what data flows through the graph
- Can be simple (dict) or complex (TypedDict, dataclass, Pydantic model)
- Modified by node functions
- Accessible to all nodes in the graph

## Nodes

Nodes are Python functions that:
- Take the current state as input
- Perform some computation
- Return updated state or partial updates

```python
def node_function(state):
    # Process state
    return {"key": "new_value"}  # Return updates
```

## Edges

Edges define transitions:
- **Normal Edge**: Simple transition from one node to another
- **Conditional Edge**: Routes based on state conditions
- **Cycles**: Create loops for iteration

## Entry and End

- **START**: Implicit starting point for graph execution
- **END**: Implicit ending point

# 3. Installation & Setup

## Prerequisites
- Python 3.9+
- pip or conda

## Installation

In [ ]:
# Install LangGraph and dependencies
# !pip install langgraph langchain langchain-core langchain-openai

# Verify installation
import langgraph
print(f"LangGraph version: {langgraph.__version__ if hasattr(langgraph, '__version__') else 'installed'}")

## Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set API keys
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# Or use .env file with: OPENAI_API_KEY=your-key-here

# 4. Building Your First Graph

## Simple Example: A Decision-Making Graph

Let's build a simple graph that:
1. Takes input text
2. Analyzes sentiment
3. Routes based on sentiment
4. Provides appropriate response

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Define state structure
class InputState(TypedDict):
    message: str
    sentiment: str
    response: str

# Create graph
graph = StateGraph(InputState)

# Define nodes
def analyze_sentiment(state):
    """Analyze the sentiment of the input message."""
    message = state["message"].lower()
    
    # Simple sentiment analysis (in practice, use an LLM)
    if any(word in message for word in ["good", "great", "amazing", "love"]):
        sentiment = "positive"
    elif any(word in message for word in ["bad", "terrible", "hate", "awful"]):
        sentiment = "negative"
    else:
        sentiment = "neutral"
    
    return {"sentiment": sentiment}

def respond_positive(state):
    """Generate response for positive sentiment."""
    return {"response": "That's wonderful! 😊 I'm glad to hear that."}

def respond_negative(state):
    """Generate response for negative sentiment."""
    return {"response": "I'm sorry to hear that. 😔 Is there anything I can help with?"}

def respond_neutral(state):
    """Generate response for neutral sentiment."""
    return {"response": "Interesting! Tell me more about that."}

# Add nodes
graph.add_node("analyze", analyze_sentiment)
graph.add_node("respond_positive", respond_positive)
graph.add_node("respond_negative", respond_negative)
graph.add_node("respond_neutral", respond_neutral)

# Add edges
graph.add_edge(START, "analyze")

# Conditional edges based on sentiment
def route_by_sentiment(state):
    sentiment = state["sentiment"]
    if sentiment == "positive":
        return "respond_positive"
    elif sentiment == "negative":
        return "respond_negative"
    else:
        return "respond_neutral"

graph.add_conditional_edges("analyze", route_by_sentiment)

# All response nodes lead to END
graph.add_edge("respond_positive", END)
graph.add_edge("respond_negative", END)
graph.add_edge("respond_neutral", END)

# Compile the graph
app = graph.compile()

print("Graph created successfully!")

## Running the Graph

In [ ]:
# Test the graph
test_inputs = [
    {"message": "This is great! I love it!", "sentiment": "", "response": ""},
    {"message": "This is terrible and awful.", "sentiment": "", "response": ""},
    {"message": "This is interesting.", "sentiment": "", "response": ""}
]

for test_input in test_inputs:
    result = app.invoke(test_input)
    print(f"\nInput: {result['message']}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Response: {result['response']}")
    print("-" * 50)

# 5. State Management

## State Definition

State defines the data schema flowing through your graph.

In [ ]:
from typing import TypedDict, List, Optional, Annotated
from operator import add

# Simple state
class SimpleState(TypedDict):
    query: str
    result: str

# Complex state with collections
class ComplexState(TypedDict):
    user_input: str
    thoughts: List[str]  # List of reasoning steps
    previous_queries: List[str]  # History of queries
    confidence_score: float
    metadata: dict

# State with reducer functions for aggregation
class AgentState(TypedDict):
    messages: Annotated[List[dict], add]  # Automatically appends to list
    iterations: int
    max_iterations: int

print("State classes defined successfully!")

## State Updates

Nodes can update state in different ways:

In [ ]:
# Example of different update patterns

# 1. Full state replacement
def full_update_node(state):
    # Returns new state values
    return {"result": "new_value"}

# 2. Partial state update
def partial_update_node(state):
    # Only updates specific fields
    return {"iterations": state.get("iterations", 0) + 1}

# 3. Using Pydantic for validation
from pydantic import BaseModel

class ValidatedState(BaseModel):
    query: str
    confidence: float
    
    class Config:
        validate_assignment = True

# 4. With operator.add for appending to lists
def append_to_list_node(state):
    # With Annotated[List, add], this appends rather than replaces
    return {"messages": [{"role": "assistant", "content": "New message"}]}

print("State update patterns demonstrated!")

# 6. Control Flow Patterns

## Linear Flow

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class LinearState(TypedDict):
    input_text: str
    processed_text: str
    analyzed_text: str

linear_graph = StateGraph(LinearState)

def process(state):
    return {"processed_text": state["input_text"].upper()}

def analyze(state):
    return {"analyzed_text": f"Analyzed: {state['processed_text']}"}

linear_graph.add_node("process", process)
linear_graph.add_node("analyze", analyze)
linear_graph.add_edge(START, "process")
linear_graph.add_edge("process", "analyze")
linear_graph.add_edge("analyze", END)

linear_app = linear_graph.compile()
print("Linear flow graph created!")

## Branching Flow

In [ ]:
class BranchState(TypedDict):
    input: str
    branch: str
    result: str

branch_graph = StateGraph(BranchState)

def router(state):
    if len(state["input"]) > 10:
        return "long_path"
    else:
        return "short_path"

def process_long(state):
    return {"branch": "long", "result": "Processed long input"}

def process_short(state):
    return {"branch": "short", "result": "Processed short input"}

branch_graph.add_node("process_long", process_long)
branch_graph.add_node("process_short", process_short)
branch_graph.add_conditional_edges(START, router)
branch_graph.add_edge("process_long", END)
branch_graph.add_edge("process_short", END)

branch_app = branch_graph.compile()
print("Branching flow graph created!")

## Cycles and Loops

In [ ]:
class LoopState(TypedDict):
    counter: int
    max_iterations: int
    results: List[str]

loop_graph = StateGraph(LoopState)

def process_iteration(state):
    counter = state["counter"]
    results = state.get("results", [])
    results.append(f"Iteration {counter}")
    return {"counter": counter + 1, "results": results}

def should_continue(state):
    # Continue if counter < max_iterations
    if state["counter"] < state["max_iterations"]:
        return "process_iteration"  # Loop back
    else:
        return "end"  # Exit loop

loop_graph.add_node("process_iteration", process_iteration)
loop_graph.add_edge(START, "process_iteration")
loop_graph.add_conditional_edges("process_iteration", should_continue)
loop_graph.add_edge("process_iteration", "process_iteration")  # Create cycle
loop_graph.add_edge("process_iteration", END)

print("Loop graph created!")

# 7. Agent Architecture

## Basic Agentic Loop

A typical agent loop contains:
1. **Agent Node**: LLM decides what action to take
2. **Tool Execution**: Execute the chosen tool
3. **Conditional Logic**: Check if done or continue loop

In [ ]:
# Example agent without actual LLM calls (for demonstration)
from typing import TypedDict, List, Literal

class Message(TypedDict):
    role: str  # "user", "assistant", "system"
    content: str

class AgentState(TypedDict):
    messages: List[Message]
    current_tool: str
    iterations: int
    max_iterations: int

# Define available tools
TOOLS = {
    "search": "Search the web for information",
    "calculate": "Perform mathematical calculations",
    "finish": "Return the final answer"
}

agent_graph = StateGraph(AgentState)

def agent_node(state):
    """LLM-based agent that decides what to do."""
    iterations = state.get("iterations", 0)
    
    # Simulate agent decision (in practice, call LLM)
    if iterations == 0:
        decision = "search"
    elif iterations == 1:
        decision = "calculate"
    else:
        decision = "finish"
    
    messages = state.get("messages", [])
    messages.append({"role": "assistant", "content": f"Using tool: {decision}"})
    
    return {"current_tool": decision, "messages": messages}

def execute_tool(state):
    """Execute the selected tool."""
    tool = state["current_tool"]
    messages = state.get("messages", [])
    
    # Simulate tool execution
    if tool == "search":
        result = "Found relevant information"
    elif tool == "calculate":
        result = "Calculation result: 42"
    else:
        result = "Task complete"
    
    messages.append({"role": "system", "content": result})
    
    return {"messages": messages, "iterations": state.get("iterations", 0) + 1}

def should_continue(state):
    """Determine if agent should continue or finish."""
    if state["current_tool"] == "finish":
        return "end"
    elif state.get("iterations", 0) >= state.get("max_iterations", 5):
        return "end"
    else:
        return "agent"

agent_graph.add_node("agent", agent_node)
agent_graph.add_node("tool", execute_tool)

agent_graph.add_edge(START, "agent")
agent_graph.add_edge("agent", "tool")
agent_graph.add_conditional_edges("tool", should_continue)
agent_graph.add_edge("tool", "agent")  # Create agentic loop

agent_app = agent_graph.compile()

print("Agent architecture created!")

## Running the Agent

In [ ]:
# Test the agent
initial_state = {
    "messages": [{"role": "user", "content": "What is 2+2?"}],
    "current_tool": "",
    "iterations": 0,
    "max_iterations": 5
}

result = agent_app.invoke(initial_state)
print("Agent execution complete!")
print(f"Final iterations: {result['iterations']}")
for msg in result["messages"]:
    print(f"{msg['role'].upper()}: {msg['content']}")

# 8. Advanced Features

## Streaming Output

In [ ]:
# Streaming results from a graph
def stream_graph_execution(graph_app, initial_state):
    """Stream execution updates."""
    for step in graph_app.stream(initial_state):
        for node, value in step.items():
            print(f"Node: {node}")
            print(f"State update: {value}")
            print("-" * 30)

# Example usage
test_state = {"input_text": "hello world", "processed_text": "", "analyzed_text": ""}
# stream_graph_execution(linear_app, test_state)

## Error Handling and Recovery

In [ ]:
class RobustState(TypedDict):
    input: str
    result: str
    error: str
    retry_count: int

robust_graph = StateGraph(RobustState)

def risky_operation(state):
    """Node that might fail."""
    try:
        # Simulate risky operation
        if "error" in state["input"]:
            raise ValueError("Intentional error")
        return {"result": f"Processed: {state['input']}"}
    except Exception as e:
        return {"error": str(e), "retry_count": state.get("retry_count", 0) + 1}

def error_handler(state):
    """Handle errors and decide on retry."""
    if state.get("retry_count", 0) < 2:
        return "retry"
    else:
        return "fail"

def retry_node(state):
    """Retry the operation."""
    return {"error": ""}  # Clear error and retry

robust_graph.add_node("operation", risky_operation)
robust_graph.add_node("retry", retry_node)
robust_graph.add_edge(START, "operation")
robust_graph.add_conditional_edges("operation", lambda s: "error_handler" if s.get("error") else "end")
robust_graph.add_conditional_edges("error_handler", error_handler)
robust_graph.add_edge("retry", "operation")

print("Error handling pattern created!")

## Human-in-the-Loop

LangGraph supports interrupting execution for human approval:

In [ ]:
# Human-in-the-loop pattern
from langgraph.graph import StateGraph, START, END

class ApprovalState(TypedDict):
    request: str
    approved: bool
    result: str

approval_graph = StateGraph(ApprovalState)

def prepare_request(state):
    """Prepare request for human review."""
    return {"request": f"Prepared: {state['request']}"}

def execute_if_approved(state):
    """Execute only if approved."""
    if state.get("approved", False):
        return {"result": "Approved and executed!"}
    else:
        return {"result": "Request was rejected."}

approval_graph.add_node("prepare", prepare_request)
approval_graph.add_node("execute", execute_if_approved)
approval_graph.add_edge(START, "prepare")
approval_graph.add_edge("prepare", "execute")  # Interrupt here for human approval
approval_graph.add_edge("execute", END)

print("Human-in-the-loop pattern created!")

# 9. Best Practices

## 1. State Design

- **Be explicit**: Use TypedDict to define exact state schema
- **Keep it minimal**: Only store what's necessary
- **Use reducers**: For aggregating data (lists, counters)
- **Document fields**: Add docstrings explaining each field

```python
class WellDesignedState(TypedDict):
    """State with clear purpose.
    
    Fields:
        user_query: The initial user input
        research_results: Accumulated research findings
        iterations: Number of processing steps completed
    """
    user_query: str
    research_results: Annotated[List[str], add]
    iterations: int
```

## 2. Node Design

- **Single responsibility**: Each node should do one thing
- **Return partial updates**: Only return changed fields
- **Handle edge cases**: Gracefully handle missing state fields
- **Use type hints**: For clarity and IDE support

```python
def focused_node(state: AgentState) -> dict:
    """Process messages and extract intent."""
    if not state.get("messages"):
        return {"error": "No messages in state"}
    # Process...
    return {"intent": extracted_intent}
```

## 3. Control Flow

- **Explicit routing**: Make decision paths clear
- **Avoid deep nesting**: Use separate conditional functions
- **Set iteration limits**: Prevent infinite loops
- **Log decisions**: Track which paths are taken

## 4. Testing and Debugging

- **Use streaming**: See intermediate results
- **Add logging**: Log node executions
- **Test edge cases**: Empty state, boundary conditions
- **Visualize graphs**: Use `graph.get_graph().draw_mermaid()` for visualization

In [ ]:
# Example of best practices
import logging
from typing import Annotated
from operator import add

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class BestPracticeState(TypedDict):
    """Well-designed application state.
    
    Tracks user requests through processing pipeline.
    """
    user_query: str
    processing_steps: Annotated[List[str], add]  # Accumulates steps
    results: List[str]
    iterations: int
    error: Optional[str]

def well_designed_node(state: BestPracticeState) -> dict:
    """Single responsibility: Process and validate.
    
    Args:
        state: Current application state
        
    Returns:
        Dictionary with updated fields only
    """
    logger.info(f"Processing query: {state.get('user_query', 'N/A')}")
    
    # Handle edge cases
    query = state.get("user_query", "")
    if not query:
        return {"error": "Empty query provided"}
    
    # Process
    result = f"Processed: {query}"
    
    # Return partial update
    return {
        "processing_steps": [{"step": "well_designed_node"}],
        "results": [result]
    }

print("Best practices example created!")

# 10. Real-World Examples

## Example 1: Research Assistant

A graph that:
1. Takes a research question
2. Plans research approach
3. Executes searches
4. Synthesizes findings
5. Returns comprehensive answer

In [ ]:
from typing import Annotated
from operator import add

class ResearchState(TypedDict):
    question: str
    research_plan: str
    searches: Annotated[List[str], add]  # Accumulate searches
    findings: Annotated[List[str], add]  # Accumulate findings
    synthesis: str
    iteration: int

research_graph = StateGraph(ResearchState)

def plan_research(state):
    """Create research plan."""
    question = state["question"]
    plan = f"Plan for researching: {question}\n1. Search for background\n2. Find recent developments\n3. Gather expert opinions"
    return {"research_plan": plan}

def execute_search(state):
    """Execute research searches."""
    iteration = state.get("iteration", 0)
    searches = [f"Search {iteration + 1}: Relevant findings"]
    findings = [f"Finding {iteration + 1}: Key insight about {state['question']}"]
    return {
        "searches": searches,
        "findings": findings,
        "iteration": iteration + 1
    }

def synthesize_findings(state):
    """Synthesize findings into answer."""
    findings = state.get("findings", [])
    synthesis = f"Based on {len(findings)} findings, here's the comprehensive answer...\n"
    for finding in findings:
        synthesis += f"- {finding}\n"
    return {"synthesis": synthesis}

research_graph.add_node("plan", plan_research)
research_graph.add_node("search", execute_search)
research_graph.add_node("synthesize", synthesize_findings)

research_graph.add_edge(START, "plan")
research_graph.add_edge("plan", "search")
research_graph.add_edge("search", "synthesize")
research_graph.add_edge("synthesize", END)

research_app = research_graph.compile()
print("Research assistant graph created!")

In [ ]:
# Test the research assistant
research_input = {
    "question": "What are the latest developments in AI safety?",
    "research_plan": "",
    "searches": [],
    "findings": [],
    "synthesis": "",
    "iteration": 0
}

research_result = research_app.invoke(research_input)
print("Research Output:")
print(research_result["synthesis"])

## Example 2: Multi-Step Verification Pipeline

A graph for validating claims through multiple verification steps:

In [ ]:
class VerificationState(TypedDict):
    claim: str
    evidence: List[str]
    verification_score: float
    verdict: str
    checks_passed: int

verification_graph = StateGraph(VerificationState)

def factual_check(state):
    """Check factual accuracy."""
    claim = state["claim"]
    # Simulate factual check
    passed = len(claim) > 10  # Simple heuristic
    evidence = [f"Factual verification: {'passed' if passed else 'failed'} for '{claim}'"]
    return {
        "evidence": evidence,
        "checks_passed": int(passed)
    }

def source_check(state):
    """Verify credible sources."""
    # Simulate source verification
    evidence = state.get("evidence", [])
    evidence.append("Source verification: checked against 5 credible sources")
    return {"evidence": evidence, "checks_passed": state.get("checks_passed", 0) + 1}

def bias_check(state):
    """Check for potential bias."""
    evidence = state.get("evidence", [])
    evidence.append("Bias assessment: no major biases detected")
    score = state.get("checks_passed", 0) / 3.0
    
    if score > 0.66:
        verdict = "VERIFIED"
    elif score > 0.33:
        verdict = "PARTIALLY VERIFIED"
    else:
        verdict = "NOT VERIFIED"
    
    return {
        "evidence": evidence,
        "verification_score": score,
        "verdict": verdict
    }

verification_graph.add_node("factual", factual_check)
verification_graph.add_node("source", source_check)
verification_graph.add_node("bias", bias_check)

verification_graph.add_edge(START, "factual")
verification_graph.add_edge("factual", "source")
verification_graph.add_edge("source", "bias")
verification_graph.add_edge("bias", END)

verification_app = verification_graph.compile()
print("Verification pipeline created!")

In [ ]:
# Test verification pipeline
verification_input = {
    "claim": "Climate change is caused by human activities and is a significant threat to our planet.",
    "evidence": [],
    "verification_score": 0.0,
    "verdict": "",
    "checks_passed": 0
}

verif_result = verification_app.invoke(verification_input)
print(f"Claim: {verif_result['claim']}")
print(f"Verdict: {verif_result['verdict']}")
print(f"Confidence Score: {verif_result['verification_score']:.2%}")
print(f"\nEvidence collected:")
for evidence in verif_result["evidence"]:
    print(f"  - {evidence}")

# Summary and Key Takeaways

## Key Concepts Recap

1. **Graphs**: Networks of nodes and edges for orchestrating workflows
2. **State**: Shared data structure flowing through the graph
3. **Nodes**: Processing units that transform state
4. **Edges**: Connections between nodes (conditional or direct)
5. **Cycles**: Enable iterative and agentic behaviors

## Core Patterns

- **Linear**: Simple step-by-step processing
- **Branching**: Different paths based on conditions
- **Looping**: Iteration with termination conditions
- **Agentic**: Agent makes decisions and uses tools
- **Error Handling**: Recovery mechanisms and retries
- **Human-in-Loop**: Pausing for human approval

## Implementation Tips

✅ Always define explicit State types  
✅ Keep nodes focused and reusable  
✅ Use conditional edges for branching logic  
✅ Implement iteration limits to prevent infinite loops  
✅ Test with streaming to see intermediate results  
✅ Add comprehensive logging  
✅ Document state transitions  

## Resources

- Official Documentation: https://langchain-ai.github.io/langgraph/
- GitHub Repository: https://github.com/langchain-ai/langgraph
- LangChain Documentation: https://python.langchain.com/

---

Happy building with LangGraph! 🚀